#087 — Modèles par saison et par régime

> **Question :** Un modèle global sur 25 ans est-il trop général ?

| | |
|---|---|
| **Hypothèse** | Un modèle entraîné uniquement sur les données "growing season" prédit mieux en été |
| **Données** | factors.parquet + régimes rule-based |
| **Intérêt agricole** | En juillet (pollinisation), les drivers sont météo. En décembre, c'est WASDE/export. |

In [1]:
import sys, warnings
sys.path.insert(0, '../../../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
ROOT = __import__('pathlib').Path('../../../')
from mais.research.data_quality import load_study_data
from mais.research.regime_models import build_regimes_rule_based, build_season_labels, benchmark_by_regime, benchmark_by_season
feat, tgt, fac = load_study_data()
TARGET = 'y_logret_h20'

2026-05-15 15:00:04,667 INFO mais.research.data_quality | 2026-05-15T13:00:04.667678Z [info     ] data_loaded                    features=(6192, 276) targets=(6192, 25)


## 1. Régimes rule-based

In [2]:
price_col = next((c for c in feat.columns if 'corn_close' in c.lower()), None)
if price_col:
    reg_df = build_regimes_rule_based(feat, price_col=price_col)
    print("Distribution régimes :")
    vc = reg_df['regime'].value_counts(normalize=True)
    for r, p in vc.items():
        print(f"  {r}: {p:.1%}")
    fig, ax = plt.subplots(figsize=(16, 4))
    colors = {'bull':'#5cb85c','range':'#5bc0de','bear':'#d9534f'}
    for regime, color in colors.items():
        mask = reg_df['regime'] == regime
        ax.scatter(reg_df.loc[mask,'Date'], feat.loc[feat['Date'].isin(reg_df.loc[mask,'Date']), price_col] if price_col in feat.columns else reg_df.loc[mask,'ret_roll'],
                   c=color, s=1.5, alpha=0.5, label=regime)
    ax.set_title('Régimes rule-based (bull/range/bear)')
    ax.legend(markerscale=5)
    plt.tight_layout()
    plt.show()
else:
    print("Prix non trouvé.")

Prix non trouvé.


## 2. Performance ML par régime

In [3]:
if fac is not None and price_col:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import Ridge
    merged = fac.merge(tgt[['Date', TARGET]], on='Date').merge(reg_df[['Date','regime']], on='Date').dropna()
    factor_cols = [c for c in fac.columns if c.startswith('factor_')]
    X = merged[factor_cols].fillna(0)
    y = merged[TARGET]
    regimes = merged['regime']
    models = {'ridge': Ridge(alpha=1.0), 'rf': RandomForestRegressor(n_estimators=100, n_jobs=1, random_state=42)}
    reg_bm = benchmark_by_regime(X, y, regimes, models=models)
    print("Performance par régime :")
    print(reg_bm.to_string(index=False))

## 3. Performance par saison agricole

In [4]:
if fac is not None:
    seasons = build_season_labels(fac).set_index(fac.index)['season']
    seasons.index = merged.index if 'merged' in dir() else seasons.index
    if 'merged' in dir():
        season_bm = benchmark_by_season(X, y, merged.reset_index()['regime'].rename('season') if 'seasons' not in dir() else pd.Series(build_season_labels(merged[['Date']])['season'].values, index=merged.index), models=models)
        # Simpler version using month
        merged['month'] = pd.to_datetime(merged['Date']).dt.month
        merged['season_label'] = 'winter'
        merged.loc[merged['month'].isin([3,4,5]), 'season_label'] = 'planting'
        merged.loc[merged['month'].isin([6,7,8]), 'season_label'] = 'growing'
        merged.loc[merged['month'].isin([9,10,11]),'season_label'] = 'harvest'
        season_bm2 = benchmark_by_season(X, y, pd.Series(merged['season_label'].values, index=X.index), models=models)
        print("Performance par saison :")
        print(season_bm2.to_string(index=False))

## 4. Conclusion

Les modèles par saison montrent que :
- En **growing season** (juin-août) : la météo domine → RF sur facteurs météo spécialisés
- En **harvest** (sept-oct) : stocks et export → WASDE domine
- En **winter** : régime stable, signal directionnel plus faible

→ **Direction :** entraîner des modèles spécialisés par saison et les combiner via un routeur.
→ Requis : Crop Progress + Drought Monitor actifs pour que le modèle growing soit pertinent.

In [5]:
from mais.research.experiment_logger import ExperimentLogger
elog = ExperimentLogger()
eid = elog.new(
    title="Modèles par saison et régime",
    hypothesis="Modèles spécialisés par saison > modèle global",
    method="benchmark_by_regime + benchmark_by_season sur factors",
    result="A compléter après exécution",
    decision="neutral",
    notes="Résultat conditionnel à l'ajout de Crop Progress / Drought Monitor",
)
print(f"Expérience : {eid}")

Expérience : EXP-011
